<a href="https://colab.research.google.com/github/aavarela/SPBD_Labs/blob/main/projeto2/project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Project 2
This work was carried out by Afonso Varela (73544) and Gonçalo Dionísio (73638), with the help of Google Gemini 3 Pro (AI agent).

#Environment configuration

In [ ]:
#@title Install & launch Kafka
%%bash
KAFKA_VERSION=3.7.2
KAFKA=kafka_2.12-$KAFKA_VERSION
wget -q -O /tmp/$KAFKA.tgz https://dlcdn.apache.org/kafka/$KAFKA_VERSION/$KAFKA.tgz
tar xfz /tmp/$KAFKA.tgz
wget -q -O $KAFKA/config/server1.properties - https://github.com/smduarte/spbd-2526/raw/refs/heads/main/docs/labs/projs/server1.properties

UUID=`$KAFKA/bin/kafka-storage.sh random-uuid`
$KAFKA/bin/kafka-storage.sh format -t $UUID -c $KAFKA/config/server1.properties
$KAFKA/bin/kafka-server-start.sh -daemon $KAFKA/config/server1.properties

In [ ]:
#@title Install pyspark 3.5.7 for compatibility with spark-sql-kafka 3.5.7
!pip uninstall -y dataproc-spark-connect
!pip uninstall -y pyspark
!pip install pyspark==3.5.7

In [ ]:
#@title Download 1% sample
!wget -q -O taxi_rides_1pc.csv.gz https://www.dropbox.com/scl/fi/v8ei5laqcalrx30z3lsty/taxi_rides_1pc.csv.gz?rlkey=q1lq7l56c4j97h9kymsdroau5&st=iurdwnwj&dl=0

In [ ]:
#@title Start Kafka publisher
!pip --quiet install kafka-python dataclasses
!wget -q -O kafka-publisher.py https://raw.githubusercontent.com/smduarte/spbd-2526/refs/heads/main/docs/labs/projs/kafka-publisher.py

!nohup python kafka-publisher.py --topic taxis_json --speedup 120 --filename taxi_rides_1pc.csv.gz 2> /dev/null &

# Data stream preparation

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession \
    .builder \
    .appName('Kafka Spark Structured Streaming Example') \
    .config('spark.jars.packages', 'org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.7') \
    .getOrCreate()

lines = spark \
  .readStream \
  .format('kafka') \
  .option('kafka.bootstrap.servers', 'localhost:9092') \
  .option('subscribe', 'taxis_json') \
  .option('startingOffsets', 'earliest') \
  .load() \
  .selectExpr('CAST(value AS STRING)')

# Define a StructType named taxi_ride_schema
taxi_ride_schema = StructType([
    StructField("medallion", StringType(), True),
    StructField("pickup_datetime", TimestampType(), True),
    StructField("dropoff_datetime", TimestampType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("pickup_latitude", DoubleType(), True),
    StructField("pickup_longitude", DoubleType(), True),
    StructField("dropoff_latitude", DoubleType(), True),
    StructField("dropoff_longitude", DoubleType(), True),
    StructField("pickup_grid_x", IntegerType(), True),
    StructField("pickup_grid_y", IntegerType(), True),
    StructField("dropoff_grid_x", IntegerType(), True),
    StructField("dropoff_grid_y", IntegerType(), True)
])

# Apply the from_json function to parse the JSON strings from lines
parsed_stream = lines.withColumn("parsed_value", from_json(col("value"), taxi_ride_schema))

# Select all fields from the parsed_value struct and project them directly
structured_stream = parsed_stream.select(col("parsed_value.*"))

MIN_LON = -74.916578
MAX_LAT = 41.47718278
LON_DELTA = 0.005986
LAT_DELTA = 0.004491556

processed_stream = structured_stream \
    .withColumn("pickup_grid_x", floor((MAX_LAT - col("pickup_latitude")) / LAT_DELTA)) \
    .withColumn("pickup_grid_y", floor((col("pickup_longitude") - MIN_LON) / LON_DELTA)) \
    .withColumn("dropoff_grid_x", floor((MAX_LAT - col("dropoff_latitude")) / LAT_DELTA)) \
    .withColumn("dropoff_grid_y", floor((col("dropoff_longitude") - MIN_LON) / LON_DELTA)) \
    .filter(
        (col("pickup_latitude").isNotNull()) & (col("pickup_longitude").isNotNull()) & \
        (col("dropoff_latitude").isNotNull()) & (col("dropoff_longitude").isNotNull()) & \
        (col("pickup_grid_x").between(0, 299)) & (col("pickup_grid_y").between(0, 299)) & \
        (col("dropoff_grid_x").between(0, 299)) & (col("dropoff_grid_y").between(0, 299)) & \
        (col("trip_distance") > 0)
    ) \
    .withColumn("trip_profit", col("fare_amount") + col("tip_amount"))

processed_stream_with_pickup_watermark = processed_stream.withWatermark("pickup_datetime", "10 minutes")
processed_stream_with_dropoff_watermark = processed_stream.withWatermark("dropoff_datetime", "10 minutes")

average_profit_per_area_stream = processed_stream_with_dropoff_watermark \
    .groupBy(
        window(col("dropoff_datetime"), "15 minutes"),
        col("pickup_grid_x"),
        col("pickup_grid_y")
    ) \
    .agg(avg("trip_profit").alias("average_trip_profit"))

dropoffs_for_empty = processed_stream_with_dropoff_watermark.select(
    col("medallion").alias("d_medallion"),
    col("dropoff_datetime").alias("d_time"),
    col("dropoff_grid_x").alias("d_grid_x"),
    col("dropoff_grid_y").alias("d_grid_y")
)

pickups_for_empty = processed_stream_with_pickup_watermark.select(
    col("medallion").alias("p_medallion"),
    col("pickup_datetime").alias("p_time"),
    col("pickup_grid_x").alias("p_grid_x"),
    col("pickup_grid_y").alias("p_grid_y")
)

empty_taxis_identification = dropoffs_for_empty.join(
    pickups_for_empty,
    expr("""
        d_medallion = p_medallion AND
        d_grid_x = p_grid_x AND
        d_grid_y = p_grid_y AND
        p_time BETWEEN d_time AND d_time + INTERVAL 30 MINUTES
    """),
    "leftOuter"
)

filtered_empty_taxis = empty_taxis_identification.filter(col("p_medallion").isNull())

empty_taxis_per_area_stream = filtered_empty_taxis \
    .groupBy(
        window(col("d_time"), "15 minutes"),
        col("d_grid_x"),
        col("d_grid_y")
    ) \
    .agg(count(col("d_medallion")).alias("empty_taxi_count"))

area_metrics_stream = average_profit_per_area_stream.join(
    empty_taxis_per_area_stream,
    (
        (average_profit_per_area_stream.window == empty_taxis_per_area_stream.window) &
        (average_profit_per_area_stream.pickup_grid_x == empty_taxis_per_area_stream.d_grid_x) &
        (average_profit_per_area_stream.pickup_grid_y == empty_taxis_per_area_stream.d_grid_y)
    ),
    "leftOuter"
).select(
    average_profit_per_area_stream.window,
    average_profit_per_area_stream.pickup_grid_x.alias("grid_x"),
    average_profit_per_area_stream.pickup_grid_y.alias("grid_y"),
    col("average_trip_profit"),
    coalesce(col("empty_taxi_count"),lit(0)).alias("empty_taxi_count")
)

area_profitability = area_metrics_stream.withColumn(
    "profitability",
    when(col("empty_taxi_count") == 0, 0.0)
    .otherwise(col("average_trip_profit") / col("empty_taxi_count"))
).select(
    col("window"),
    col("grid_x"),
    col("grid_y"),
    col("profitability")
)

# Collect area profitability data

In [ ]:
import time
import pandas as pd

# Stop any previous active queries
for q in spark.streams.active:
    q.stop()
print("All currently active streaming queries stopped.")

# Start a streaming query to collect the data
data_query = area_profitability \
    .writeStream \
    .outputMode("append") \
    .format("memory") \
    .queryName("collected_data") \
    .start()

print("Streaming query started.")

print("Waiting to allow data to be collected.")
time.sleep(900)

# Stop the streaming query
data_query.stop()

print("Streaming query stopped. Data collection complete.")

# Retrieve the collected data from the memory sink into a Pandas DataFrame
area_profitability_df = spark.sql("SELECT * FROM collected_data").toPandas()

# Display the collected data
print("Displaying the first 5 rows of the collected data:")
display(area_profitability_df.head())

print(f"Total rows in area_profitability data: {len(area_profitability_df)}")

# 1. Analysis of the area profitability data

In [ ]:
#@title Show a heat map of the area profitability data
import matplotlib.pyplot as plt
import seaborn as sns

# Aggregate profitability by grid coordinates (average over time windows)
aggregated_profitability = area_profitability_df.groupby(['grid_x', 'grid_y'])['profitability'].mean().reset_index()

# Pivot the data for heatmap
heatmap_data = aggregated_profitability.pivot_table(index='grid_x', columns='grid_y', values='profitability')

plt.figure(figsize=(12, 10))
sns.heatmap(heatmap_data, cmap='viridis', annot=False, fmt=".1f", cbar_kws={'label': 'Average Profitability'})
plt.title('Average Profitability by Grid Area')
plt.xlabel('Grid Y')
plt.ylabel('Grid X')
plt.gca().invert_yaxis() # Invert y-axis to have (0,0) at top-left, matching typical grid representation
plt.show()

In [ ]:
#@title Show some key summary statistics of the area profitability data
print("Key Summary Statistics for Average Profitability:")
display(aggregated_profitability['profitability'].describe())

In [ ]:
#@title Show histogram of the area profitability data
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram
plt.figure(figsize=(10, 6))
sns.histplot(area_profitability_df['profitability'], bins=50)
plt.title('Distribution of Area Profitability')
plt.xlabel('Profitability')
plt.ylabel('Frequency')
plt.show()


## Observations:
*   **Distribution Shape**: The histogram shows a skewed distribution, heavily concentrated towards lower profitability values, including a significant number of areas with zero profitability.
*   **Concentration at Zero**: A large peak at profitability = 0 indicates many grid areas likely had no trips or no successful trips during the observed time windows, or perhaps the formula resulted in zero for other reasons.
*   **Range**: While many areas have low profitability, there's a long tail extending to higher profitability values, suggesting a few highly profitable areas.
*   **Interpretation**: This implies that profitability is not uniformly distributed across the grid. There are specific areas that are very profitable, while a substantial number of areas yield little to no profit.

In [ ]:
#@title Show the box plot of the area profitability data
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.boxplot(x=area_profitability_df['profitability'])
plt.title('Box Plot of Area Profitability')
plt.xlabel('Profitability')
plt.show()

## Observations:
*   **Median and Quartiles**: The box plot clearly indicates that the median profitability is very low, likely at or near zero. The first quartile (Q1) and third quartile (Q3) are also low, reinforcing the observation from the histogram that most areas have low profitability.
*   **Outliers**: The box plot shows a considerable number of individual data points extending far above the upper whisker. These represent significant outliers – areas with exceptionally high profitability compared to the majority.
*   **Spread**: The interquartile range (IQR) is relatively small, indicating that the bulk of the data (the middle 50%) is tightly clustered at lower profitability values.
*   **Interpretation**: The box plot visually confirms the presence of highly profitable hotspots (outliers) that stand out from the vast majority of areas, which exhibit very low or zero profitability. This suggests that identifying and focusing on these high-profitability areas could be crucial for optimizing taxi operations.

## Analysis key findings
*   The profitability distribution is heavily skewed, with a significant concentration of areas showing low or zero profitability.
*   A prominent peak at profitability = 0 suggests many grid areas had no trips or no successful trips, or yield no profit.
*   While most areas have low profitability, there is a long tail extending to higher values, indicating the presence of a few exceptionally profitable areas.
*   The median profitability is very low, close to zero, and the first and third quartiles are also low, reinforcing that the majority of areas are not highly profitable.
*   The box plot clearly identifies a considerable number of outliers, representing areas with exceptionally high profitability, significantly above the median and the bulk of the data.
*   The interquartile range (IQR) is relatively small, showing that the central 50% of areas are clustered at very low profitability levels.

# 2. How does the profitability depend on the trip distance?

In [ ]:
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Stop any previous active queries
for q in spark.streams.active:
    q.stop()
print("All currently active streaming queries stopped.")

# Select relevant columns for this analysis
trip_data_stream = processed_stream.select("trip_distance", "trip_profit")

# Start a streaming query to collect the data into a memory sink
trip_data_query = trip_data_stream \
    .writeStream \
    .outputMode("append") \
    .format("memory") \
    .queryName("collected_trip_data") \
    .start()

print("Streaming query for trip data started.")

print("Waiting to allow trip data to be collected (900 seconds)...")
time.sleep(900)

# Stop the streaming query
trip_data_query.stop()

print("Streaming query for trip data stopped. Data collection complete.")

# Retrieve the collected data from the memory sink into a Pandas DataFrame
trip_profit_distance_df = spark.sql("SELECT * FROM collected_trip_data").toPandas()

# Display the collected data
print("Displaying the first 5 rows of the collected trip data:")
display(trip_profit_distance_df.head())

print(f"Total rows in trip_profit_distance_df: {len(trip_profit_distance_df)}")

# Group by trip_distance and calculate the average trip_profit
# Binning distances can help if there are too many unique values
trip_profit_distance_df['distance_bin'] = pd.cut(trip_profit_distance_df['trip_distance'], bins=50, precision=1)

# Calculate the average profitability for each distance bin
averaged_by_distance = trip_profit_distance_df.groupby('distance_bin')['trip_profit'].mean().reset_index()

# Convert interval to string for plotting
averaged_by_distance['distance_bin_str'] = averaged_by_distance['distance_bin'].astype(str)

plt.figure(figsize=(14, 7))
sns.lineplot(x='distance_bin_str', y='trip_profit', data=averaged_by_distance)
plt.title('Average Trip Profit by Trip Distance Bins')
plt.xlabel('Trip Distance Bin')
plt.ylabel('Average Trip Profit')
plt.xticks(rotation=90)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

# Also show a scatter plot for a more detailed view (can be very dense)
plt.figure(figsize=(14, 7))
sns.scatterplot(x='trip_distance', y='trip_profit', data=trip_profit_distance_df, alpha=0.3, s=10)
plt.title('Trip Profit vs. Trip Distance (Raw Data)')
plt.xlabel('Trip Distance')
plt.ylabel('Trip Profit')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

### Observations:
*   **General Trend**: There appears to be a general positive correlation between trip distance and trip profit, meaning longer trips tend to yield higher profits.
*   **Variability**: For any given trip distance, there's a significant variability in `trip_profit`, especially for shorter distances, likely due to varying `fare_amount` and `tip_amount`.
*   **Outliers/High Profit for Short Distances**: There might be instances of very high profits even for relatively short distances, possibly due to high surge pricing or generous tips. These appear as outliers in the scatter plot.
*   **Concentration at Lower Distances**: The majority of trips seem to be shorter, as indicated by the density of points on the left side of the scatter plot.
*   **Diminishing Returns (potentially)**: While profit generally increases with distance, the rate of increase might slow down for very long distances, or the number of such trips might be fewer, making the average less robust. The line plot suggests a relatively linear increase for most bins, with some fluctuations.

#3. How does the profitability depend on the trip time of day

# Addendum

# Task
Analyze how trip profitability depends on the time of day by collecting `pickup_datetime` and `trip_profit` data from the `processed_stream`, extracting the hour of the day, calculating the average trip profit for each hour, visualizing this relationship with a line plot, and summarizing the key observations.

## Collect Trip Data with Time of Day and Profit

### Subtask:
Create a new streaming query to collect `pickup_datetime` and `trip_profit` from the `processed_stream` into a memory sink.


**Reasoning**:
The subtask requires collecting `pickup_datetime` and `trip_profit` from the `processed_stream` into a memory sink, converting it to a Pandas DataFrame, and displaying its head and row count. This involves stopping existing queries, initiating a new streaming query, waiting for data, stopping the query, and then retrieving and displaying the data, which can all be done in a single code block.



In [ ]:
import time
import pandas as pd

# Stop any previous active queries
for q in spark.streams.active:
    q.stop()
print("All currently active streaming queries stopped.")

# Select relevant columns for this analysis
trip_time_data_stream = processed_stream.select("pickup_datetime", "trip_profit")

# Start a streaming query to collect the data into a memory sink
trip_time_data_query = trip_time_data_stream \
    .writeStream \
    .outputMode("append") \
    .format("memory") \
    .queryName("collected_time_data") \
    .start()

print("Streaming query for time data started.")

print("Waiting to allow time data to be collected (900 seconds)...")
time.sleep(900)

# Stop the streaming query
trip_time_data_query.stop()

print("Streaming query for time data stopped. Data collection complete.")

# Retrieve the collected data from the memory sink into a Pandas DataFrame
trip_profit_time_df = spark.sql("SELECT * FROM collected_time_data").toPandas()

# Display the collected data
print("Displaying the first 5 rows of the collected time data:")
display(trip_profit_time_df.head())

print(f"Total rows in trip_profit_time_df: {len(trip_profit_time_df)}")

**Reasoning**:
Now that the data has been collected, the next logical step according to the task is to extract the hour from the 'pickup_datetime' column and then calculate the average trip profit for each hour. This will prepare the data for visualization and analysis of profitability based on time of day.



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Extract the hour from 'pickup_datetime'
trip_profit_time_df['pickup_hour'] = trip_profit_time_df['pickup_datetime'].dt.hour

# Calculate the average trip profit for each hour
average_profit_per_hour = trip_profit_time_df.groupby('pickup_hour')['trip_profit'].mean().reset_index()

# Sort by hour for proper plotting
average_profit_per_hour = average_profit_per_hour.sort_values('pickup_hour')

# Display the first few rows of the aggregated data
print("Average trip profit per hour:")
display(average_profit_per_hour.head())


**Reasoning**:
To visualize the relationship between the hour of the day and average trip profitability, a line plot is the most suitable choice as it effectively shows trends over a continuous variable (time).



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create a line plot to visualize average trip profit by hour
plt.figure(figsize=(12, 6))
sns.lineplot(x='pickup_hour', y='trip_profit', data=average_profit_per_hour, marker='o')
plt.title('Average Trip Profit by Hour of Day')
plt.xlabel('Hour of Day (24-hour format)')
plt.ylabel('Average Trip Profit')
plt.xticks(range(0, 24)) # Ensure all hours are displayed on the x-axis
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

### Observations:
*   **Evening Peak**: The plot shows a significant peak in average trip profitability during the evening hours, specifically around 18:00 (6 PM). This suggests higher demand, potentially higher fares, or better tipping during this period.
*   **Decline after Peak**: After the evening peak, profitability tends to decline steadily through the late evening and night, reaching its lowest points in the early morning hours.
*   **Morning/Daytime Lows**: Profitability appears relatively low during the early morning and throughout the day, indicating less lucrative trips or lower demand compared to the evening peak.
*   **Interpretation**: This pattern suggests that taxi drivers might find it more profitable to work during the evening rush or post-work hours, while early mornings and daytime might be less profitable. Strategic deployment of taxis could be optimized based on these hourly profitability trends.

## Summary:

### Q&A
Trip profitability heavily depends on the time of day, showing a significant peak around 18:00 (6 PM) with an average profit of approximately \$45.99. Profitability generally declines after this evening peak, reaching lower levels during late evening, night, and early morning hours.

### Data Analysis Key Findings
*   A total of 54,291 rows of `pickup_datetime` and `trip_profit` data were collected for analysis.
*   The average trip profit shows a distinct peak at 18:00 (6 PM), with an average profit of approximately \$45.99.
*   Following the 18:00 peak, trip profitability tends to decline steadily through the late evening and night, reaching lower values such as \$14.98 at 19:00 and \$3.87 at 22:00.
*   Profitability is notably lower during early morning and daytime hours compared to the evening peak.

### Insights or Next Steps
*   **Optimal Driving Hours**: Taxi drivers or ride-sharing platforms could optimize their operational hours by concentrating efforts during the evening peak (e.g., around 18:00) to maximize earnings.
*   **Dynamic Pricing/Incentives**: Ride-sharing companies could implement dynamic pricing or driver incentives during historically low-profit hours (e.g., early morning, daytime) to balance driver availability and demand.
